# **AI Hub 데이터 사용 실습 (Windows / macOS 로컬 PC)**


---



## **실습 환경 준비**

- (선택) Visual Studio Code를 열고,
- (터미널:Ctrl+j에서) 가상환경을 만들고
- (확장:Ctrl+Shift+X) Jupyter를 설치한 뒤 이 노트북을 연다.
- 아래 셀은 현재 노트북 커널에 실습 패키지를 설치한다.

| 운영체제 | 터미널에서 최초 1회 실행 |
|---|---|
| Windows PowerShell | `py -m venv .venv` → `.\.venv\Scripts\Activate.ps1` → `python -m pip install jupyter` → `jupyter notebook` |
| macOS 터미널 | `python3 -m venv .venv` → `source .venv/bin/activate` → `python -m pip install jupyter` → `jupyter notebook` |

In [ ]:
%pip install -q -U numpy pandas matplotlib pillow scikit-learn scikit-image joblib requests tqdm

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_candidates = {
    "Windows": ["Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"],
    "Darwin": ["Apple SD Gothic Neo", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"],
}.get(platform.system(), ["NanumGothic", "Noto Sans CJK KR"])

installed_fonts = {font.name for font in fm.fontManager.ttflist}
font_name = next((name for name in font_candidates if name in installed_fonts), None)

if font_name:
    plt.rcParams["font.family"] = font_name
    print(f"한글 폰트: {font_name}")
else:
    print("한글 폰트를 찾지 못했습니다. Windows는 맑은 고딕, macOS는 AppleGothic을 확인하세요.")

plt.rcParams["font.size"] = 12
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (6, 4)

**환경 확인**

In [ ]:
import os, sys, json, re, shutil, zipfile, random, math, platform
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# 원하는 저장 위치로 바꿀 수 있습니다. 예: Path(r"D:\aihub_plant_project")
PROJECT_DIR = (Path.cwd() / "aihub_plant_project").resolve()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("운영체제:", platform.platform())
print("Python:", sys.version.split()[0])
print("프로젝트 폴더:", PROJECT_DIR)
print("디스크 여유 공간(GB):", round(shutil.disk_usage(PROJECT_DIR).free / 1024**3, 2))

assert sys.version_info >= (3, 10)
print("✔ 환경 확인 완료")



---



## **예제: 이미지 데이터 전처리 실습**

- **데이터셋:** AIHub `식물 병 유발 통합 데이터` (`dataSetSn=525`)  
    - https://aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&dataSetSn=525
- **환경:** Windows 10/11 또는 macOS의 Jupyter Notebook / JupyterLab / VS Code  
- **목표:** AIHub의 **Training 데이터**를 사용하여 다음 전 과정을 실습한다.

> AIHub Training ZIP 선택 다운로드 → ZIP 구조 확인 → JSON 라벨 파싱 → JPG/JSON 매칭 →  
> 클래스별 subset 추출 → 이미지 검증 → Resize/Normalize → Train/Validation/Test 분리 →  
> HOG + Linear SVM 학습 → 평가

-  왜 이렇게 구성하는가?
    - AIHub의 `filekey` 선택 다운로드는 **파일(ZIP) 단위**이다.  
    - 즉 AIHub 서버에서 이미지 600장만 직접 골라 내려받을 수는 없다.
    - 따라서 이 노트북은 다음 전략을 사용한다.

    1. 데이터셋 525의 파일 목록을 조회한다.
    2. `1.Training/원천데이터` 중 **병해 이미지 ZIP 후보**를 자동 탐색한다.
    3. 후보 가운데 가장 작은 ZIP을 기본값으로 추천한다.
    4. 대응되는 Training 라벨 ZIP도 자동 탐색한다.
    5. ZIP 전체를 디스크에 풀지 않고 내부 파일을 읽는다.
    6. 실제 학습에는 클래스별 최대 300장만 추출한다.

    > **주의:** AIHub 데이터 다운로드 승인이 완료되어 있어야 하며 개인 API Key가 필요하다.  
    > AIHub는 Windows에서 공식 Shell 사용 시 WSL을 안내하지만, 이 노트북은 HTTP 다운로드·TAR 해제·분할 ZIP 병합을 Python으로 구현하여 Windows/macOS에서 동일하게 실행한다.




---



### **1. 라이브러리 버전 확인**

In [ ]:
import sklearn, skimage, PIL, matplotlib
print("scikit-learn:", sklearn.__version__)
print("scikit-image:", skimage.__version__)
print("Pillow:", PIL.__version__)
print("✔ 라이브러리 준비 완료")

### **2. 운영체제 독립형 AIHub 다운로드 함수 준비**

In [ ]:
import tarfile
import requests
from tqdm.auto import tqdm

AIHUB_BASE_URL = "https://api.aihub.or.kr"
AIHUB_DOWNLOAD_VERSION = "0.6"

def get_aihub_file_tree(dataset_key):
    # AIHub API는 정상 본문을 주면서도 502 등을 반환하는 경우가 있어
    # 상태 코드 대신 본문 내용으로 성공 여부를 판단한다.
    url = f"{AIHUB_BASE_URL}/info/{dataset_key}.do"
    response = requests.get(url, timeout=60)
    response.encoding = response.apparent_encoding or "utf-8"
    text = response.text

    if ".zip" not in text.lower():
        raise RuntimeError(
            "AIHub 파일 목록을 받지 못했습니다. "
            f"(HTTP {response.status_code}) {text[:500]}"
        )
    return text

def download_with_resume(url, headers, target_path, chunk_size=8 * 1024**2):
    target_path = Path(target_path)
    partial_path = target_path.with_suffix(target_path.suffix + ".partial")
    downloaded = partial_path.stat().st_size if partial_path.exists() else 0
    request_headers = dict(headers)
    if downloaded:
        request_headers["Range"] = f"bytes={downloaded}-"

    with requests.get(url, headers=request_headers, stream=True, timeout=(30, 300)) as response:
        # 다운로드 API도 502를 돌려주면서 정상 전송하는 사례가 있어
        # 인증/권한 오류(4xx)만 즉시 예외로 처리한다.
        if 400 <= response.status_code < 500:
            response.raise_for_status()
        resume_ok = downloaded > 0 and response.status_code == 206
        if not resume_ok:
            downloaded = 0
        mode = "ab" if resume_ok else "wb"
        remaining = int(response.headers.get("Content-Length", 0))
        total = downloaded + remaining if remaining else None
        with partial_path.open(mode) as file, tqdm(
            total=total, initial=downloaded, unit="B", unit_scale=True,
            desc="AIHub 다운로드"
        ) as progress:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))

    partial_path.replace(target_path)
    return target_path

def safe_extract_tar(tar_path, target_dir):
    target_dir = Path(target_dir).resolve()
    with tarfile.open(tar_path) as archive:
        for member in archive.getmembers():
            if member.issym() or member.islnk():
                raise RuntimeError(f"TAR 링크 항목은 허용하지 않습니다: {member.name}")
            destination = (target_dir / member.name).resolve()
            if os.path.commonpath([str(target_dir), str(destination)]) != str(target_dir):
                raise RuntimeError(f"안전하지 않은 TAR 경로: {member.name}")
        archive.extractall(target_dir)

def merge_split_files(root_dir):
    groups = defaultdict(list)
    for part in Path(root_dir).rglob("*.part*"):
        match = re.search(r"\.part(\d+)$", part.name)
        if match:
            groups[part.parent / part.name[:match.start()]].append(
                (int(match.group(1)), part)
            )

    for merged_path, numbered_parts in groups.items():
        numbered_parts.sort(key=lambda item: item[0])
        print(f"분할 파일 병합: {merged_path.name} ({len(numbered_parts)}개)")
        with merged_path.open("wb") as merged:
            for _, part in numbered_parts:
                with part.open("rb") as source:
                    shutil.copyfileobj(source, merged, length=8 * 1024**2)
        for _, part in numbered_parts:
            part.unlink()

def download_aihub_files(dataset_key, filekeys, api_key, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    tar_path = target_dir / "download.tar"
    file_sn = ",".join(map(str, filekeys))
    url = f"{AIHUB_BASE_URL}/down/{AIHUB_DOWNLOAD_VERSION}/{dataset_key}.do?fileSn={file_sn}"
    download_with_resume(url, {"apikey": api_key}, tar_path)

    if not tarfile.is_tarfile(tar_path):
        with tar_path.open("rb") as file:
            message = file.read(1000).decode("utf-8", errors="replace")
        raise RuntimeError(f"AIHub 응답이 TAR 파일이 아닙니다. 승인/API Key를 확인하세요.\n{message}")

    print("TAR 해제 중...")
    safe_extract_tar(tar_path, target_dir)
    tar_path.unlink()  # 대용량 임시 TAR 삭제
    merge_split_files(target_dir)
    return target_dir

print("✔ Windows/macOS 공통 다운로드 함수 준비 완료")

### **3. AIHub API Key 입력**

API Key를 코드에 직접 저장하지 않고 실행 중에만 입력한다.

In [ ]:
from getpass import getpass

AIHUB_API_KEY = getpass("AIHub API Key 입력: ").strip()

assert len(AIHUB_API_KEY) >= 10, "API Key가 너무 짧습니다."
print("✔ API Key 입력 완료")

### **4. 데이터셋 525의 파일 목록 조회**

In [ ]:
DATASET_KEY = "525"

file_tree_text = get_aihub_file_tree(DATASET_KEY)
print(file_tree_text[:12000])

assert "Training" in file_tree_text, (
    "Training 파일 목록을 찾지 못했습니다. "
    "AIHub 다운로드 승인 상태와 datasetkey를 확인하세요."
)
print("✔ 데이터셋 파일 목록 조회 완료")

### **5. Training ZIP 후보 자동 추출**

In [ ]:
SIZE_FACTORS = {
    "B": 1,
    "KB": 1024,
    "MB": 1024**2,
    "GB": 1024**3,
    "TB": 1024**4,
}

def size_to_bytes(size_text):
    m = re.search(r"([\d.]+)\s*(B|KB|MB|GB|TB)", str(size_text), re.I)
    if not m:
        return np.nan
    return float(m.group(1)) * SIZE_FACTORS[m.group(2).upper()]

def parse_aihub_file_tree(text):
    rows = []

    for raw in text.splitlines():
        line = raw.strip()

        if "|" not in line or ".zip" not in line.lower():
            continue

        parts = [p.strip() for p in line.split("|")]

        if len(parts) < 3:
            continue

        left = re.sub(r"^[│├└─\s]+", "", parts[0]).strip()
        filename = Path(left).name.strip()
        size_text = parts[1].strip()

        m_key = re.search(r"(\d+)", parts[2])
        if not m_key:
            continue

        rows.append({
            "line": line,
            "filename": filename,
            "size": size_text,
            "size_bytes": size_to_bytes(size_text),
            "filekey": m_key.group(1),
        })

    return pd.DataFrame(rows)

files_df = parse_aihub_file_tree(file_tree_text)

assert len(files_df) > 0, "ZIP 파일 정보를 파싱하지 못했습니다."

display(files_df[["filename", "size", "filekey"]].head(30))
print("ZIP 파일 수:", len(files_df))
print("✔ 파일 목록 파싱 완료")

### **6. Training 원천데이터 중 `병해` ZIP 선택**

기본값은 가장 작은 `TS...병해...zip`이다.

자동 선택이 적절하지 않으면 `SOURCE_FILEKEY`를 직접 지정한다.

In [ ]:
SOURCE_FILEKEY = 51654   # 예: "12345" # 원천: 토마토 병해
LABEL_FILEKEY  = 51639   # 예: "12346" # 라벨: 토마토
MAX_DOWNLOAD_GB = 42.0  # 실수로 더 큰 ZIP을 선택하지 않도록 제한

source_candidates = files_df[
    files_df["filename"].str.match(r"(?i)^TS")
    & files_df["filename"].str.contains("병해", na=False)
].copy()

if source_candidates.empty:
    source_candidates = files_df[
        files_df["filename"].str.match(r"(?i)^TS")
    ].copy()

source_candidates = source_candidates.sort_values(
    "size_bytes", na_position="last"
).reset_index(drop=True)

display(source_candidates[["filename", "size", "filekey"]].head(20))

assert len(source_candidates) > 0, "Training 원천데이터(TS) 후보가 없습니다."

if SOURCE_FILEKEY is None:
    chosen_source = source_candidates.iloc[0]
else:
    hit = source_candidates[source_candidates["filekey"] == str(SOURCE_FILEKEY)]
    assert len(hit) == 1, "지정한 SOURCE_FILEKEY를 후보 목록에서 찾지 못했습니다."
    chosen_source = hit.iloc[0]

print("선택된 Training 원천 ZIP")
print("파일명 :", chosen_source["filename"])
print("용량   :", chosen_source["size"])
print("filekey:", chosen_source["filekey"])

if pd.notna(chosen_source["size_bytes"]):
    size_gb = chosen_source["size_bytes"] / 1024**3
    assert size_gb <= MAX_DOWNLOAD_GB, (
        f"선택 ZIP이 {size_gb:.1f}GB로 MAX_DOWNLOAD_GB={MAX_DOWNLOAD_GB}GB를 초과합니다.\n"
        "더 작은 Training ZIP을 선택하거나 MAX_DOWNLOAD_GB를 조정하세요."
    )

print("✔ Training 원천 ZIP 선택 완료")

### **7. 대응 Training 라벨 ZIP 자동 선택**

In [ ]:
def guess_crop_name(filename):
    name = Path(filename).stem
    chunks = re.split(r"[_\-\s]+", name)
    chunks = [re.sub(r"(?i)^TS\d*", "", c).strip() for c in chunks]
    chunks = [
        c for c in chunks
        if c and c not in {"병해", "생리장해", "작물보호제처리반응"}
    ]
    return chunks[0] if chunks else None

crop_name = guess_crop_name(chosen_source["filename"])
print("추정 작물명:", crop_name)

label_candidates = files_df[
    files_df["filename"].str.match(r"(?i)^TL")
].copy()

if LABEL_FILEKEY is not None:
    hit = label_candidates[label_candidates["filekey"] == str(LABEL_FILEKEY)]
    assert len(hit) == 1, "지정한 LABEL_FILEKEY를 찾지 못했습니다."
    chosen_label = hit.iloc[0]
else:
    if crop_name:
        crop_hits = label_candidates[
            label_candidates["filename"].str.contains(
                str(crop_name),
                regex=False,
                na=False
            )
        ]
    else:
        crop_hits = pd.DataFrame()

    if len(crop_hits) > 0:
        chosen_label = crop_hits.sort_values(
            "size_bytes",
            na_position="last"
        ).iloc[0]
    else:
        print("⚠ 대응 라벨 ZIP 자동 탐색 실패. 아래 TL 후보를 확인하세요.")
        display(label_candidates[["filename", "size", "filekey"]])
        raise RuntimeError(
            "LABEL_FILEKEY를 직접 지정한 뒤 이 셀부터 다시 실행하세요."
        )

print("선택된 Training 라벨 ZIP")
print("파일명 :", chosen_label["filename"])
print("용량   :", chosen_label["size"])
print("filekey:", chosen_label["filekey"])
print("✔ Training 라벨 ZIP 선택 완료")

### **8. 다운로드 전 로컬 디스크 공간 검증**

In [ ]:
free_bytes = shutil.disk_usage(PROJECT_DIR).free

source_bytes = (
    float(chosen_source["size_bytes"])
    if pd.notna(chosen_source["size_bytes"])
    else 0
)

label_bytes = (
    float(chosen_label["size_bytes"])
    if pd.notna(chosen_label["size_bytes"])
    else 0
)

# 다운로드 TAR, 압축 해제된 분할 파일, 병합 ZIP이 잠시 함께 존재할 수 있다.
need_bytes = (source_bytes + label_bytes) * 2.5

print("현재 여유 공간:", round(free_bytes / 1024**3, 2), "GB")
print("권장 최소 공간:", round(need_bytes / 1024**3, 2), "GB")

if source_bytes > 0:
    assert free_bytes > need_bytes, (
        "로컬 디스크 공간이 부족할 가능성이 큽니다. "
        "더 작은 Training ZIP을 선택하세요."
    )

print("✔ 다운로드 공간 검증 통과")

### **9. AIHub Training 원천 ZIP + 라벨 ZIP 다운로드**

In [ ]:
DOWNLOAD_DIR = PROJECT_DIR / "aihub525"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

filekeys = [chosen_source["filekey"], chosen_label["filekey"]]
print("다운로드 대상 filekey:", ",".join(map(str, filekeys)))

expected_names = {Path(chosen_source["filename"]).name, Path(chosen_label["filename"]).name}
existing_names = {path.name for path in DOWNLOAD_DIR.rglob("*.zip")}

if expected_names.issubset(existing_names):
    print("✔ 필요한 ZIP이 이미 있어 다운로드를 건너뜁니다.")
else:
    download_aihub_files(
        dataset_key=DATASET_KEY,
        filekeys=filekeys,
        api_key=AIHUB_API_KEY,
        target_dir=DOWNLOAD_DIR,
    )
    print("✔ 다운로드·TAR 해제·분할 ZIP 병합 완료")

### **10. 다운로드된 ZIP 자동 탐색**

In [ ]:
all_zips = list(DOWNLOAD_DIR.rglob("*.zip"))

print("발견된 ZIP 수:", len(all_zips))
for p in all_zips[:30]:
    print(" -", p, f"({p.stat().st_size / 1024**2:.1f} MB)")

assert len(all_zips) >= 2, "원천 ZIP과 라벨 ZIP을 찾지 못했습니다."

def find_zip_by_name(filename):
    exact = [p for p in all_zips if p.name == filename]

    if exact:
        return exact[0]

    stem = Path(filename).stem
    partial = [p for p in all_zips if stem in p.stem]

    return partial[0] if partial else None

SOURCE_ZIP = find_zip_by_name(chosen_source["filename"])
LABEL_ZIP = find_zip_by_name(chosen_label["filename"])

assert SOURCE_ZIP is not None, "Training 원천 ZIP을 찾지 못했습니다."
assert LABEL_ZIP is not None, "Training 라벨 ZIP을 찾지 못했습니다."

print("SOURCE_ZIP:", SOURCE_ZIP)
print("LABEL_ZIP :", LABEL_ZIP)
print("✔ ZIP 탐색 완료")

### **11. ZIP을 풀지 않고 내부 구조 확인**

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

with zipfile.ZipFile(SOURCE_ZIP) as z:
    image_members = [
        n for n in z.namelist()
        if Path(n).suffix.lower() in IMAGE_EXTS and not n.endswith("/")
    ]

with zipfile.ZipFile(LABEL_ZIP) as z:
    json_members = [
        n for n in z.namelist()
        if Path(n).suffix.lower() == ".json" and not n.endswith("/")
    ]

print("원천 이미지 수:", len(image_members))
print("JSON 라벨 수 :", len(json_members))
print("이미지 예시  :", image_members[:3])
print("JSON 예시    :", json_members[:3])

assert len(image_members) > 0, "원천 ZIP에 이미지가 없습니다."
assert len(json_members) > 0, "라벨 ZIP에 JSON이 없습니다."
print("✔ ZIP 내부 구조 검증 완료")

### **12. JSON 라벨 파싱**

In [ ]:
def parse_label_obj(obj, json_member):
    desc = obj.get("description", {}) if isinstance(obj, dict) else {}
    # AIHub 525 라벨 스키마의 키는 "annotations"(복수)다.
    # 데이터셋 버전에 따라 "annotation"인 경우도 있어 둘 다 허용한다.
    ann = {}
    if isinstance(obj, dict):
        ann = obj.get("annotations") or obj.get("annotation") or {}
    if not isinstance(ann, dict):
        ann = {}

    image_name = desc.get("image")

    return {
        "json_member": json_member,
        "image_name": Path(str(image_name)).name if image_name else None,
        "image_stem": Path(str(image_name)).stem if image_name else None,
        "crop": ann.get("crop"),
        "disease": ann.get("disease"),
        "risk": ann.get("risk"),
        "bbox": ann.get("bbox"),
    }

label_rows = []

with zipfile.ZipFile(LABEL_ZIP) as z:
    for member in json_members:
        try:
            raw = z.read(member)
            obj = json.loads(raw.decode("utf-8-sig"))
            label_rows.append(parse_label_obj(obj, member))
        except Exception as e:
            label_rows.append({
                "json_member": member,
                "image_name": None,
                "image_stem": None,
                "crop": None,
                "disease": None,
                "risk": None,
                "bbox": None,
                "error": str(e)
            })

labels_df = pd.DataFrame(label_rows)

print("라벨 행 수:", len(labels_df))
display(labels_df.head())

assert "disease" in labels_df.columns
assert labels_df["image_name"].notna().sum() > 0
assert labels_df["disease"].notna().sum() > 0, (
    "disease 값을 하나도 읽지 못했습니다. JSON 스키마 키를 확인하세요."
)
print("✔ JSON 라벨 파싱 완료")

### **13. JPG ↔ JSON 매칭**

In [ ]:
image_by_name = {Path(n).name: n for n in image_members}
image_by_stem = {Path(n).stem: n for n in image_members}

def match_image(row):
    name = row.get("image_name")
    stem = row.get("image_stem")

    if name in image_by_name:
        return image_by_name[name]

    if stem in image_by_stem:
        return image_by_stem[stem]

    return None

labels_df["image_member"] = labels_df.apply(match_image, axis=1)

matched_df = labels_df[
    labels_df["image_member"].notna()
    & labels_df["disease"].notna()
].copy()

print("JSON 전체:", len(labels_df))
print("이미지 매칭:", len(matched_df))
print("매칭률: %.2f%%" % (100 * len(matched_df) / max(1, len(labels_df))))

display(
    matched_df["disease"]
    .value_counts(dropna=False)
    .rename("count")
    .to_frame()
    .head(20)
)

assert len(matched_df) > 0, "JPG와 JSON 매칭에 실패했습니다."
print("✔ JPG ↔ JSON 매칭 완료")

### **14. 학습용 Training subset 만들기**

- 정상 코드 `00` 제외
- 가장 많은 질병 클래스 2개 선택
- 클래스별 최대 300장
- 최대 600장

In [ ]:
N_PER_CLASS = 300
N_CLASSES = 2
RANDOM_STATE = 42

work_df = matched_df.copy()
work_df["disease"] = work_df["disease"].astype(str)

disease_df = work_df[
    work_df["disease"] != "00"
].copy()

counts = disease_df["disease"].value_counts()

print("질병 클래스 분포:")
display(counts.to_frame("count").head(20))

assert len(counts) >= N_CLASSES, (
    f"질병 클래스가 {N_CLASSES}개 미만입니다. "
    "다른 Training 병해 ZIP을 선택하세요."
)

selected_classes = counts.head(N_CLASSES).index.tolist()
print("선택 클래스:", selected_classes)

subset_parts = []

for cls in selected_classes:
    g = disease_df[disease_df["disease"] == cls]
    n = min(N_PER_CLASS, len(g))

    subset_parts.append(
        g.sample(
            n=n,
            random_state=RANDOM_STATE
        )
    )

subset_df = (
    pd.concat(subset_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print("subset 크기:", len(subset_df))
display(subset_df["disease"].value_counts().to_frame("count"))

assert subset_df["disease"].nunique() == N_CLASSES
assert len(subset_df) >= 100, "실습용 데이터가 너무 적습니다."
print("✔ Training subset 생성 완료")

### **15. subset 이미지만 압축 해제**

대용량 ZIP 전체를 풀지 않고 선택된 subset만 추출한다.

In [ ]:
from PIL import Image

SUBSET_DIR = PROJECT_DIR / "plant_subset"
IMAGE_DIR = SUBSET_DIR / "images"
LABEL_DIR = SUBSET_DIR / "labels"

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
LABEL_DIR.mkdir(parents=True, exist_ok=True)

for p in IMAGE_DIR.iterdir():
    if p.is_file():
        p.unlink()

for p in LABEL_DIR.iterdir():
    if p.is_file():
        p.unlink()

with zipfile.ZipFile(SOURCE_ZIP) as src_zip, zipfile.ZipFile(LABEL_ZIP) as lbl_zip:

    for row in subset_df.itertuples():

        img_name = Path(row.image_member).name
        json_name = Path(row.json_member).name

        (IMAGE_DIR / img_name).write_bytes(
            src_zip.read(row.image_member)
        )

        (LABEL_DIR / json_name).write_bytes(
            lbl_zip.read(row.json_member)
        )

print("추출 이미지:", len(list(IMAGE_DIR.glob("*"))))
print("추출 JSON :", len(list(LABEL_DIR.glob("*.json"))))

assert len(list(IMAGE_DIR.glob("*"))) == len(subset_df)
assert len(list(LABEL_DIR.glob("*.json"))) == len(subset_df)

print("✔ subset 파일 추출 완료")

### **16. 이미지 무결성 검사**

In [ ]:
from PIL import Image

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

valid_flags = [
    (p.name, is_valid_image(p))
    for p in IMAGE_DIR.iterdir()
    if p.is_file()
]

invalid = [
    name for name, ok in valid_flags
    if not ok
]

print("검사 이미지:", len(valid_flags))
print("손상 이미지:", len(invalid))
print("손상 예시:", invalid[:10])

assert len(invalid) == 0, "손상 이미지가 있습니다."
print("✔ 이미지 무결성 검사 통과")

### **17. Resize + Normalize**

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def load_rgb_normalized(path, size=(128, 128)):

    with Image.open(path) as img:
        img = img.convert("RGB").resize(size)
        arr = np.asarray(
            img,
            dtype=np.float32
        ) / 255.0

    return arr

sample_path = IMAGE_DIR / Path(
    subset_df.iloc[0]["image_member"]
).name

sample_img = load_rgb_normalized(sample_path)

print("shape:", sample_img.shape)
print("dtype:", sample_img.dtype)
print("min/max:", sample_img.min(), sample_img.max())

assert sample_img.shape == (128, 128, 3)
assert 0.0 <= sample_img.min() <= sample_img.max() <= 1.0

plt.figure(figsize=(4, 4))
plt.imshow(sample_img)
plt.axis("off")
plt.title("Preprocessed sample")
plt.show()

print("✔ Resize / Normalize 검증 완료")

### **18. HOG 특징 추출**

In [ ]:
from skimage.color import rgb2gray
from skimage.feature import hog

def extract_hog_from_path(path):

    rgb = load_rgb_normalized(
        path,
        size=(128, 128)
    )

    gray = rgb2gray(rgb)

    feat = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
    )

    return feat.astype(np.float32)

X_list = []
y_list = []

for row in subset_df.itertuples():

    img_path = IMAGE_DIR / Path(
        row.image_member
    ).name

    X_list.append(
        extract_hog_from_path(img_path)
    )

    y_list.append(
        str(row.disease)
    )

X = np.vstack(X_list)
y = np.asarray(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature 수:", X.shape[1])

assert X.ndim == 2
assert len(X) == len(y)
assert np.isfinite(X).all()

print("✔ 이미지 → HOG feature matrix 변환 완료")

### **18-1. 샘플 이미지 HOG 특징 시각화**

8100차원 feature 벡터가 실제로 무엇을 담고 있는지 샘플 1장으로 확인한다.


In [ ]:
from skimage import exposure

def extract_hog_with_image(path):
    """셀 18과 동일한 설정으로 HOG를 뽑되, 시각화용 이미지도 함께 반환한다."""
    rgb = load_rgb_normalized(path, size=(128, 128))
    gray = rgb2gray(rgb)

    feat, hog_image = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
        visualize=True,
    )

    return rgb, gray, feat.astype(np.float32), hog_image

VIZ_INDEX = 0   # 다른 샘플을 보려면 이 번호를 바꾼다 (0 ~ len(subset_df)-1)

viz_row = subset_df.iloc[VIZ_INDEX]
viz_path = IMAGE_DIR / Path(viz_row["image_member"]).name

rgb, gray, feat, hog_image = extract_hog_with_image(viz_path)

# 셀 18에서 만든 X의 같은 행과 동일한 벡터인지 확인
assert feat.shape[0] == X.shape[1]
assert np.allclose(feat, X[VIZ_INDEX], atol=1e-5)

print("파일명        :", viz_path.name)
print("클래스        :", viz_row["disease"])
print("feature 차원  :", feat.shape[0])
print("min / max     :", round(float(feat.min()), 4), "/", round(float(feat.max()), 4))
print("mean / std    :", round(float(feat.mean()), 4), "/", round(float(feat.std()), 4))
print("0에 가까운 값 비율:", round(float((feat < 0.01).mean()) * 100, 1), "%")

# 1) 원본 -> grayscale -> HOG gradient 방향 이미지
hog_image_vis = exposure.rescale_intensity(hog_image, in_range=(0, hog_image.max()))

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

axes[0].imshow(rgb)
axes[0].set_title("① 원본 128x128")
axes[0].set_xlabel("클래스: " + str(viz_row["disease"]))

axes[1].imshow(gray, cmap="gray")
axes[1].set_title("② Grayscale")

axes[2].imshow(hog_image_vis, cmap="gray")
axes[2].set_title("③ HOG gradient 방향")

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

# 2) feature 벡터 자체의 모습
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(feat, linewidth=0.5)
axes[0].set_title("HOG feature 벡터 (" + str(feat.shape[0]) + "차원)")
axes[0].set_xlabel("feature index")
axes[0].set_ylabel("값")

axes[1].hist(feat, bins=50)
axes[1].set_yscale("log")
axes[1].set_title("feature 값 분포")
axes[1].set_xlabel("값")
axes[1].set_ylabel("개수 (log)")

# feature 벡터는 (블록..., cells_per_block..., orientations=9) 순서로 펼쳐져 있다
orientation_mean = feat.reshape(-1, 9).mean(axis=0)
bin_centers = (np.arange(9) + 0.5) * (180 / 9)

axes[2].bar(bin_centers, orientation_mean, width=15)
axes[2].set_title("방향 bin별 평균 세기")
axes[2].set_xlabel("gradient 방향 (도)")
axes[2].set_ylabel("평균값")

plt.tight_layout()
plt.show()

print("가장 강한 방향:", round(float(bin_centers[orientation_mean.argmax()]), 1), "도")
print("✔ 샘플 HOG 특징 시각화 완료")


### **19. Train / Validation / Test 분할**

원본은 모두 AIHub **Training 데이터**이고,  
그 내부 subset을 분할기준에 따라 70/15/15로 다시 나눈다.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train     :", X_train.shape, Counter(y_train))
print("Validation:", X_val.shape, Counter(y_val))
print("Test      :", X_test.shape, Counter(y_test))

assert len(X_train) + len(X_val) + len(X_test) == len(X)
assert set(y_train) == set(y_val) == set(y_test)

print("✔ Train / Validation / Test 분할 완료")

### **20. Linear SVM 학습**

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

model = make_pipeline(
    StandardScaler(),
    LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=42,
        max_iter=10000
    )
)

model.fit(X_train, y_train)

print("✔ 모델 학습 완료")

### **21. Validation 성능 확인**

In [ ]:
from sklearn.metrics import (
    classification_report,
    f1_score,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

val_pred = model.predict(X_val)

print(
    classification_report(
        y_val,
        val_pred,
        zero_division=0
    )
)

print(
    "Validation Macro-F1:",
    round(
        f1_score(
            y_val,
            val_pred,
            average="macro"
        ),
        4
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_val,
    val_pred
)
plt.title("Validation Confusion Matrix")
plt.show()

print("✔ Validation 평가 완료")

### **22. 최종 Test 평가**

In [ ]:
test_pred = model.predict(X_test)

test_f1 = f1_score(
    y_test,
    test_pred,
    average="macro"
)

print(
    classification_report(
        y_test,
        test_pred,
        zero_division=0
    )
)

print(
    "Test Macro-F1:",
    round(test_f1, 4)
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred
)
plt.title("Test Confusion Matrix")
plt.show()

assert 0.0 <= test_f1 <= 1.0
print("✔ 최종 Test 평가 완료")

### **23. 전처리 결과와 모델 저장**

In [ ]:
import joblib

OUT_DIR = PROJECT_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

subset_csv = (
    OUT_DIR
    / "plant_training_subset_metadata.csv"
)

model_path = (
    OUT_DIR
    / "hog_linear_svm.joblib"
)

subset_df[
    [
        "image_name",
        "crop",
        "disease",
        "risk",
        "image_member",
        "json_member"
    ]
].to_csv(
    subset_csv,
    index=False,
    encoding="utf-8-sig"
)

joblib.dump(
    model,
    model_path
)

print("metadata:", subset_csv)
print("model   :", model_path)

loaded = joblib.load(model_path)

assert np.array_equal(
    loaded.predict(X_test[:10]),
    model.predict(X_test[:10])
)

print("✔ 저장/재로딩 검증 완료")

---

## **[실습]: 저장된 모델로 새 이미지 평가하기**

23번에서 `output/hog_linear_svm.joblib` 에 저장한 모델을 다시 불러와,
**학습에 한 번도 사용하지 않은 이미지**로 성능을 확인한다.

### **24. 미학습(holdout) 이미지 준비**

- `subset_df`(= Train/Validation/Test 에 사용된 600장)에 **포함되지 않은** 이미지만 골라 `output/holdout/` 에 저장한다.
- `파일명 → 정답 라벨` 조회표도 `output/label_lookup.csv` 로 저장한다.
  이렇게 해 두면 나중에 노트북을 다시 열었을 때 24~27번 셀만 실행해도 평가가 가능하다.

In [ ]:
HOLDOUT_PER_CLASS = 40   # 클래스별로 뽑을 미학습 이미지 수

OUT_DIR = PROJECT_DIR / "output"
HOLDOUT_DIR = OUT_DIR / "holdout"
HOLDOUT_DIR.mkdir(parents=True, exist_ok=True)

used_members = set(subset_df["image_member"])

# 학습에 쓰인 이미지를 제외하고, 모델이 아는 클래스만 남긴다.
holdout_pool = matched_df[
    matched_df["disease"].astype(str).isin(selected_classes)
    & ~matched_df["image_member"].isin(used_members)
].copy()

print("전체 매칭 이미지 :", len(matched_df))
print("학습에 쓰인 이미지:", len(used_members))
print("미학습 후보      :", len(holdout_pool))

holdout_parts = []

for cls in selected_classes:
    g = holdout_pool[holdout_pool["disease"].astype(str) == cls]
    n = min(HOLDOUT_PER_CLASS, len(g))
    if n > 0:
        holdout_parts.append(g.sample(n=n, random_state=RANDOM_STATE))

holdout_df = pd.concat(holdout_parts, ignore_index=True)

# 폴더를 비우고 다시 추출 (여러 번 실행해도 안전)
for p in HOLDOUT_DIR.glob("*"):
    if p.is_file():
        p.unlink()

with zipfile.ZipFile(SOURCE_ZIP) as src_zip:
    for row in holdout_df.itertuples():
        img_name = Path(row.image_member).name
        (HOLDOUT_DIR / img_name).write_bytes(src_zip.read(row.image_member))

# 파일명 -> 정답 라벨 조회표 (학습 사용 여부 포함)
lookup_df = matched_df[["crop", "disease", "risk", "image_member"]].copy()
lookup_df["image_name"] = lookup_df["image_member"].map(lambda m: Path(m).name)
lookup_df["disease"] = lookup_df["disease"].astype(str)
lookup_df["used_in_training"] = lookup_df["image_member"].isin(used_members)

LOOKUP_CSV = OUT_DIR / "label_lookup.csv"
lookup_df[
    ["image_name", "crop", "disease", "risk", "used_in_training"]
].to_csv(LOOKUP_CSV, index=False, encoding="utf-8-sig")

print("holdout 폴더 :", HOLDOUT_DIR)
print("holdout 이미지:", len(list(HOLDOUT_DIR.glob("*"))), "장")
display(holdout_df["disease"].value_counts().to_frame("count"))
print("라벨 조회표  :", LOOKUP_CSV, f"({len(lookup_df)}행)")

assert len(holdout_df) > 0, "미학습 이미지를 만들지 못했습니다."
assert holdout_df["image_member"].isin(used_members).sum() == 0, "학습 이미지가 섞였습니다."

print("✔ 미학습 이미지 준비 완료")

### **25. 저장된 모델 불러오기 + 단일 이미지 추론 함수**

- 23번에서 저장한 `hog_linear_svm.joblib` 을 다시 읽는다.
- 학습 때와 **완전히 같은 전처리**(RGB → 128×128 → /255 → grayscale → HOG)를 거쳐야 한다.
- `LinearSVC` 는 확률을 내지 않으므로, 결정경계까지의 거리(margin)를 참고용 신뢰도로 환산해 보여 준다.

In [ ]:
import io
import base64
import joblib
from PIL import Image
from skimage import exposure
from skimage.color import rgb2gray
from skimage.feature import hog

OUT_DIR = PROJECT_DIR / "output"
HOLDOUT_DIR = OUT_DIR / "holdout"
MODEL_PATH = OUT_DIR / "hog_linear_svm.joblib"
LOOKUP_CSV = OUT_DIR / "label_lookup.csv"
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
IMG_SIZE = (128, 128)

clf = joblib.load(MODEL_PATH)
CLASSES = [str(c) for c in clf.classes_]

# 파일명 -> 정답 라벨 / 학습 사용 여부
truth_lookup = {}
train_used = set()

if LOOKUP_CSV.exists():
    lookup_loaded = pd.read_csv(LOOKUP_CSV, dtype=str).fillna("")
    for r in lookup_loaded.itertuples():
        truth_lookup[str(r.image_name)] = str(r.disease)
        if str(r.used_in_training).strip().lower() in {"true", "1"}:
            train_used.add(str(r.image_name))

def hog_from_pil(img, size=IMG_SIZE):
    """학습 때와 동일한 전처리 + HOG. 시각화용 HOG 이미지도 함께 반환한다."""
    rgb = np.asarray(img.convert("RGB").resize(size), dtype=np.float32) / 255.0
    gray = rgb2gray(rgb)

    feat, hog_image = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True,
        visualize=True,
    )

    return rgb, feat.astype(np.float32), hog_image

def predict_pil(img):
    """PIL 이미지 1장 -> 예측 결과 dict"""
    rgb, feat, hog_image = hog_from_pil(img)
    x = feat.reshape(1, -1)

    pred = str(clf.predict(x)[0])
    raw = clf.decision_function(x)[0]

    if np.ndim(raw) == 0:                      # 2-클래스: 스칼라 1개
        margin = float(abs(raw))
        scores = {CLASSES[0]: float(-raw), CLASSES[1]: float(raw)}
    else:                                      # 다중 클래스
        ordered = np.sort(np.asarray(raw, dtype=float))
        margin = float(ordered[-1] - ordered[-2])
        scores = {c: float(s) for c, s in zip(CLASSES, np.asarray(raw, dtype=float))}

    return {
        "pred": pred,
        "margin": margin,
        # 확률이 아니라 "결정경계에서 얼마나 떨어져 있는가"를 0.5~1.0으로 환산한 참고값
        "confidence": float(1.0 / (1.0 + math.exp(-margin))),
        "scores": scores,
        "rgb": rgb,
        "hog_image": hog_image,
    }

print("모델 경로   :", MODEL_PATH)
print("모델 클래스 :", CLASSES)
print("라벨 조회표 :", len(truth_lookup), "건")

# 동작 확인: holdout 이미지 1장
demo_paths = sorted(p for p in HOLDOUT_DIR.glob("*") if p.suffix.lower() in IMAGE_EXTS)
assert demo_paths, "holdout 이미지가 없습니다. 24번 셀을 먼저 실행하세요."

with Image.open(demo_paths[0]) as im:
    demo = predict_pil(im)

print("샘플 파일   :", demo_paths[0].name)
print("정답 라벨   :", truth_lookup.get(demo_paths[0].name, "(없음)"))
print("예측 라벨   :", demo["pred"], "| margin:", round(demo["margin"], 3),
      "| 신뢰도:", round(demo["confidence"], 3))

print("✔ 추론 함수 준비 완료")

### **26. 미학습 이미지 폴더 일괄 평가**

`output/holdout/` 전체를 예측해 **Test 성능과 비교**한다.
값이 Test 성능과 비슷하면 모델이 잘 일반화된 것이고, 크게 낮으면 과적합을 의심한다.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    ConfusionMatrixDisplay,
)

holdout_paths = sorted(p for p in HOLDOUT_DIR.glob("*") if p.suffix.lower() in IMAGE_EXTS)

records = []

for p in holdout_paths:
    with Image.open(p) as im:
        result = predict_pil(im)

    records.append({
        "file": p.name,
        "true": truth_lookup.get(p.name),
        "pred": result["pred"],
        "confidence": round(result["confidence"], 3),
        "margin": round(result["margin"], 3),
    })

holdout_pred_df = pd.DataFrame(records)
# 정답 라벨이 없는 이미지는 True/False 대신 None 으로 둔다.
holdout_pred_df["correct"] = [
    (t == p) if isinstance(t, str) else None
    for t, p in zip(holdout_pred_df["true"], holdout_pred_df["pred"])
]

evaluable = holdout_pred_df[holdout_pred_df["true"].notna()]

print("평가 이미지:", len(holdout_pred_df), "장 (정답 라벨 있음:", len(evaluable), "장)")
display(holdout_pred_df.head(10))

holdout_f1 = f1_score(evaluable["true"], evaluable["pred"], average="macro")

print(classification_report(evaluable["true"], evaluable["pred"], zero_division=0))
print("미학습 Accuracy:", round(accuracy_score(evaluable["true"], evaluable["pred"]), 4))
print("미학습 Macro-F1:", round(holdout_f1, 4))
if globals().get("test_f1") is not None:
    print("(참고) Test Macro-F1:", round(test_f1, 4))

ConfusionMatrixDisplay.from_predictions(evaluable["true"], evaluable["pred"])
plt.title("Holdout(미학습) Confusion Matrix")
plt.show()

# 틀린 샘플 몇 장 눈으로 확인
wrong = evaluable[evaluable["correct"] == False].head(5)

if len(wrong) > 0:
    fig, axes = plt.subplots(1, len(wrong), figsize=(3.2 * len(wrong), 3.6))
    axes = np.atleast_1d(axes)

    for ax, row in zip(axes, wrong.itertuples()):
        with Image.open(HOLDOUT_DIR / row.file) as im:
            ax.imshow(im.convert("RGB").resize(IMG_SIZE))
        ax.set_title(f"정답 {row.true} → 예측 {row.pred}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

    plt.suptitle("오분류 샘플")
    plt.tight_layout()
    plt.show()
else:
    print("오분류 샘플이 없습니다.")

print("✔ 미학습 이미지 일괄 평가 완료")

### **27. 웹 업로드 데모 (마우스로 이미지를 올리면 시각화 + 평가 보고서)**

- 노트북 안에서 작은 **Flask 웹 서버**를 띄운다.
- 브라우저에서 이미지를 **드래그하거나 클릭해서 선택**하면
  - 원본 / HOG 시각화
  - 예측 클래스와 신뢰도
  - (파일명이 `label_lookup.csv` 에 있으면) 정답과 비교한 **평가 보고서**

  가 함께 표시된다.
- 여러 장을 한 번에 올리면 Accuracy · Macro-F1 · 클래스별 성능 · 혼동행렬이 계산된다.
- 테스트용 이미지는 `output/holdout/` 폴더에 들어 있다. 파일 탐색기에서 그 폴더를 열고 여러 장을 선택해 끌어다 놓으면 된다.

In [ ]:
%pip install -q flask

import threading
import socket
import logging

from flask import Flask, request, jsonify, Response
from werkzeug.serving import make_server
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

logging.getLogger("werkzeug").setLevel(logging.ERROR)

def pil_to_data_url(img, max_side=320):
    """PIL 이미지를 HTML <img> 에 바로 넣을 수 있는 base64 문자열로 바꾼다."""
    thumb = img.convert("RGB").copy()
    thumb.thumbnail((max_side, max_side))
    buffer = io.BytesIO()
    thumb.save(buffer, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode("ascii")

def array_to_data_url(arr):
    """HOG 시각화 배열(0~1 실수)을 이미지로 바꾼다."""
    a = np.asarray(arr, dtype=np.float32)
    top = float(a.max()) if float(a.max()) > 0 else 1.0
    a = exposure.rescale_intensity(a, in_range=(0.0, top))
    return pil_to_data_url(Image.fromarray((a * 255).astype(np.uint8)).convert("RGB"))

PAGE = """<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>식물 병해 분류 데모</title>
<style>
  :root { --line:#e3e6ea; --ink:#1f2933; --muted:#6b7684; --ok:#0f9d58; --bad:#d93025; --bg:#f6f7f9; }
  * { box-sizing:border-box; }
  body { margin:0; padding:24px 16px 64px; background:var(--bg); color:var(--ink);
         font-family:"Malgun Gothic","Apple SD Gothic Neo","Noto Sans KR",sans-serif; }
  .wrap { max-width:1040px; margin:0 auto; }
  h1 { font-size:22px; margin:0 0 4px; }
  h2 { font-size:16px; margin:0 0 12px; }
  .sub { color:var(--muted); font-size:14px; margin-bottom:20px; }
  .card { background:#fff; border:1px solid var(--line); border-radius:12px; padding:20px; margin-bottom:18px; }
  #drop { border:2px dashed #b9c0c9; border-radius:12px; background:#fff; padding:40px 20px;
          text-align:center; cursor:pointer; transition:.15s; }
  #drop.hover { border-color:#1a73e8; background:#eef4ff; }
  #drop .big { font-size:16px; font-weight:600; }
  #drop .small { color:var(--muted); font-size:13px; margin-top:6px; word-break:break-all; }
  .row { display:flex; gap:10px; align-items:center; margin-top:14px; flex-wrap:wrap; }
  button { background:#1a73e8; color:#fff; border:0; border-radius:8px; padding:10px 18px;
           font-size:14px; cursor:pointer; }
  button.ghost { background:#fff; color:var(--ink); border:1px solid var(--line); }
  button:disabled { opacity:.5; cursor:default; }
  .tiles { display:flex; gap:12px; flex-wrap:wrap; }
  .tile { flex:1 1 150px; border:1px solid var(--line); border-radius:10px; padding:14px; }
  .tile .k { font-size:12px; color:var(--muted); }
  .tile .v { font-size:24px; font-weight:700; margin-top:4px; }
  table { border-collapse:collapse; width:100%; font-size:14px; }
  th, td { border:1px solid var(--line); padding:8px 10px; text-align:center; }
  th { background:#f2f4f7; font-weight:600; }
  td.head { background:#f2f4f7; font-weight:600; }
  .grid { display:grid; grid-template-columns:repeat(auto-fill,minmax(300px,1fr)); gap:14px; }
  .item { border:1px solid var(--line); border-radius:10px; overflow:hidden; background:#fff; }
  .item .imgs { display:flex; background:#000; }
  .item .imgs img { width:50%; display:block; aspect-ratio:1/1; object-fit:cover; }
  .item .body { padding:12px; font-size:13px; }
  .item .name { font-weight:600; word-break:break-all; margin-bottom:6px; }
  .badge { display:inline-block; border-radius:999px; padding:2px 10px; font-size:12px; color:#fff; }
  .badge.ok { background:var(--ok); } .badge.bad { background:var(--bad); } .badge.na { background:#9aa0a6; }
  .kv { color:var(--muted); margin-top:4px; }
  .bar { height:6px; background:#e8eaed; border-radius:4px; margin-top:6px; overflow:hidden; }
  .bar > span { display:block; height:100%; background:#1a73e8; }
  .note { color:var(--muted); font-size:12px; margin-top:10px; line-height:1.7; }
</style>
</head>
<body>
<div class="wrap">
  <h1>식물 병해 이미지 분류 데모</h1>
  <div class="sub">HOG + Linear SVM &middot; 학습 클래스: <b id="classes">...</b></div>

  <div class="card">
    <div id="drop">
      <div class="big">이미지를 여기에 끌어다 놓거나, 클릭해서 선택하세요</div>
      <div class="small">JPG / PNG, 여러 장 동시 선택 가능<br>예시 폴더: <span id="holdout"></span></div>
    </div>
    <input id="picker" type="file" accept="image/*" multiple hidden>
    <div class="row">
      <button id="run" disabled>평가 실행</button>
      <button id="clear" class="ghost">선택 초기화</button>
      <span id="status" class="sub" style="margin:0"></span>
    </div>
  </div>

  <div id="out"></div>
</div>

<script>
const drop = document.getElementById('drop');
const picker = document.getElementById('picker');
const runBtn = document.getElementById('run');
const clearBtn = document.getElementById('clear');
const statusEl = document.getElementById('status');
const out = document.getElementById('out');
let files = [];

fetch('/api/info').then(r => r.json()).then(d => {
  document.getElementById('classes').textContent = d.classes.join(', ');
  document.getElementById('holdout').textContent = d.holdout_dir;
});

function setFiles(list) {
  files = Array.from(list).filter(f => f.type.indexOf('image/') === 0);
  runBtn.disabled = files.length === 0;
  statusEl.textContent = files.length ? files.length + '장 선택됨' : '';
}

drop.addEventListener('click', () => picker.click());
picker.addEventListener('change', e => setFiles(e.target.files));
['dragenter', 'dragover'].forEach(ev => drop.addEventListener(ev, e => {
  e.preventDefault();
  drop.classList.add('hover');
}));
['dragleave', 'drop'].forEach(ev => drop.addEventListener(ev, e => {
  e.preventDefault();
  drop.classList.remove('hover');
}));
drop.addEventListener('drop', e => setFiles(e.dataTransfer.files));
clearBtn.addEventListener('click', () => {
  picker.value = '';
  setFiles([]);
  out.innerHTML = '';
});

runBtn.addEventListener('click', async () => {
  const form = new FormData();
  files.forEach(f => form.append('images', f, f.name));
  runBtn.disabled = true;
  statusEl.textContent = '예측 중... (' + files.length + '장)';
  try {
    const res = await fetch('/api/predict', { method: 'POST', body: form });
    render(await res.json());
    statusEl.textContent = '완료';
  } catch (err) {
    statusEl.textContent = '오류: ' + err;
  }
  runBtn.disabled = false;
});

function pct(x) { return (100 * x).toFixed(1) + '%'; }

function tile(k, v) {
  return '<div class="tile"><div class="k">' + k + '</div><div class="v">' + v + '</div></div>';
}

function render(data) {
  const s = data.summary;
  let html = '';

  html += '<div class="card"><h2>평가 요약</h2><div class="tiles">';
  html += tile('업로드 이미지', s.total + '장');
  html += tile('정답 라벨 확인', s.labeled + '장');
  html += tile('Accuracy', s.labeled ? pct(s.accuracy) : '-');
  html += tile('Macro-F1', s.labeled ? s.macro_f1.toFixed(4) : '-');
  html += '</div><div class="note">';
  html += '정답 라벨은 파일명을 label_lookup.csv 에서 찾아 붙입니다. ';
  if (s.unlabeled) { html += '라벨을 찾지 못한 ' + s.unlabeled + '장은 지표 계산에서 제외했습니다. '; }
  if (s.unknown_class) { html += '모델이 배우지 않은 클래스 ' + s.unknown_class + '장도 제외했습니다. '; }
  if (s.trained) { html += '<b style="color:#d93025">주의: ' + s.trained + '장은 학습에 사용된 이미지라 성능이 과대평가됩니다.</b>'; }
  html += '</div></div>';

  if (data.report.length) {
    html += '<div class="card"><h2>클래스별 성능</h2><table>';
    html += '<tr><th>클래스</th><th>Precision</th><th>Recall</th><th>F1</th><th>Support</th></tr>';
    data.report.forEach(r => {
      html += '<tr><td class="head">' + r.label + '</td><td>' + r.precision.toFixed(3) +
              '</td><td>' + r.recall.toFixed(3) + '</td><td>' + r.f1.toFixed(3) +
              '</td><td>' + r.support + '</td></tr>';
    });
    html += '</table>';

    const cm = data.confusion;
    html += '<h2 style="margin-top:22px">혼동행렬 (행: 정답, 열: 예측)</h2><table><tr><th></th>';
    cm.labels.forEach(l => { html += '<th>예측 ' + l + '</th>'; });
    html += '</tr>';
    cm.matrix.forEach((cells, i) => {
      html += '<tr><td class="head">정답 ' + cm.labels[i] + '</td>';
      cells.forEach((v, j) => {
        const style = i === j ? 'background:#e6f4ea;font-weight:700' : (v ? 'background:#fce8e6' : '');
        html += '<td style="' + style + '">' + v + '</td>';
      });
      html += '</tr>';
    });
    html += '</table></div>';
  }

  html += '<div class="card"><h2>이미지별 결과 (왼쪽: 입력 128x128, 오른쪽: HOG 특징)</h2><div class="grid">';
  data.items.forEach(it => {
    if (it.error) {
      html += '<div class="item"><div class="body"><div class="name">' + it.name +
              '</div><span class="badge bad">읽기 실패</span><div class="kv">' + it.error + '</div></div></div>';
      return;
    }
    let badge = '<span class="badge na">정답 라벨 없음</span>';
    if (it.true_label !== null) {
      badge = it.correct ? '<span class="badge ok">정답</span>' : '<span class="badge bad">오답</span>';
    }
    html += '<div class="item">' +
      '<div class="imgs"><img src="' + it.image + '"><img src="' + it.hog + '"></div>' +
      '<div class="body"><div class="name">' + it.name + '</div>' + badge +
      '<div class="kv">예측: <b>' + it.pred + '</b>' +
      (it.true_label !== null ? ' / 정답: <b>' + it.true_label + '</b>' : '') + '</div>' +
      '<div class="kv">신뢰도 ' + pct(it.confidence) + ' (margin ' + it.margin.toFixed(3) + ')</div>' +
      '<div class="bar"><span style="width:' + pct(it.confidence) + '"></span></div>' +
      (it.used_in_training ? '<div class="kv" style="color:#d93025">학습에 사용된 이미지</div>' : '') +
      '</div></div>';
  });
  html += '</div><div class="note">신뢰도는 확률이 아니라, SVM 결정경계까지의 거리(margin)를 0.5~1.0 사이 값으로 환산한 참고값입니다.</div></div>';

  out.innerHTML = html;
  out.scrollIntoView({ behavior: 'smooth', block: 'start' });
}
</script>
</body>
</html>
"""

app = Flask(__name__)

@app.route("/")
def index():
    return Response(PAGE, mimetype="text/html")

@app.route("/api/info")
def api_info():
    return jsonify({
        "classes": CLASSES,
        "model_path": str(MODEL_PATH),
        "holdout_dir": str(HOLDOUT_DIR),
    })

@app.route("/api/predict", methods=["POST"])
def api_predict():
    items = []

    for storage in request.files.getlist("images"):
        name = Path(storage.filename or "unknown").name

        try:
            with Image.open(io.BytesIO(storage.read())) as opened:
                image = opened.convert("RGB")
        except Exception as exc:
            items.append({"name": name, "error": str(exc)})
            continue

        result = predict_pil(image)
        truth = truth_lookup.get(name)

        items.append({
            "name": name,
            "pred": result["pred"],
            "true_label": truth,
            "correct": (truth == result["pred"]) if truth is not None else None,
            "confidence": result["confidence"],
            "margin": result["margin"],
            "scores": result["scores"],
            "used_in_training": name in train_used,
            "image": pil_to_data_url(image.resize(IMG_SIZE)),
            "hog": array_to_data_url(result["hog_image"]),
        })

    ok_items = [it for it in items if not it.get("error")]

    # 정답 라벨이 있고, 모델이 학습한 클래스인 것만 지표 계산에 쓴다.
    scored = [
        it for it in ok_items
        if it["true_label"] is not None and it["true_label"] in CLASSES
    ]

    summary = {
        "total": len(items),
        "labeled": len(scored),
        "unlabeled": sum(1 for it in ok_items if it["true_label"] is None),
        "unknown_class": sum(
            1 for it in ok_items
            if it["true_label"] is not None and it["true_label"] not in CLASSES
        ),
        "trained": sum(1 for it in ok_items if it["used_in_training"]),
        "accuracy": 0.0,
        "macro_f1": 0.0,
    }

    report_rows = []
    confusion = {"labels": [], "matrix": []}

    if scored:
        y_true = [it["true_label"] for it in scored]
        y_pred = [it["pred"] for it in scored]

        summary["accuracy"] = float(accuracy_score(y_true, y_pred))
        summary["macro_f1"] = float(f1_score(y_true, y_pred, average="macro", zero_division=0))

        detail = classification_report(
            y_true,
            y_pred,
            labels=CLASSES,
            output_dict=True,
            zero_division=0,
        )

        for key in CLASSES + ["macro avg", "weighted avg"]:
            if key in detail:
                row = detail[key]
                report_rows.append({
                    "label": key,
                    "precision": float(row["precision"]),
                    "recall": float(row["recall"]),
                    "f1": float(row["f1-score"]),
                    "support": int(row["support"]),
                })

        confusion = {
            "labels": CLASSES,
            "matrix": confusion_matrix(y_true, y_pred, labels=CLASSES).tolist(),
        }

    return jsonify({
        "items": items,
        "summary": summary,
        "report": report_rows,
        "confusion": confusion,
    })

def find_free_port(preferred=7860):
    for port in range(preferred, preferred + 20):
        with socket.socket() as probe:
            if probe.connect_ex(("127.0.0.1", port)) != 0:
                return port
    return 0

# 이 셀을 여러 번 실행해도 서버가 중복으로 뜨지 않도록 이전 서버를 정리한다.
if globals().get("WEB_SERVER") is not None:
    WEB_SERVER.shutdown()
    WEB_SERVER.server_close()

WEB_PORT = find_free_port()
WEB_SERVER = make_server("127.0.0.1", WEB_PORT, app)
WEB_THREAD = threading.Thread(target=WEB_SERVER.serve_forever, daemon=True)
WEB_THREAD.start()

WEB_URL = f"http://127.0.0.1:{WEB_PORT}/"

print("웹 데모 주소      :", WEB_URL)
print("테스트 이미지 폴더:", HOLDOUT_DIR)
print("서버를 멈추려면 28번 셀을 실행한다. (커널을 종료해도 함께 멈춘다)")

try:
    import webbrowser
    webbrowser.open(WEB_URL)
except Exception as exc:
    print("브라우저 자동 실행 실패. 위 주소를 직접 열어라:", exc)

### **28. 웹 서버 종료**

실습이 끝나면 포트를 반납한다. (노트북 커널을 종료해도 함께 멈춘다)

In [ ]:
if globals().get("WEB_SERVER") is not None:
    WEB_SERVER.shutdown()
    WEB_SERVER.server_close()
    WEB_SERVER = None
    print("✔ 웹 서버를 종료했습니다.")
else:
    print("실행 중인 웹 서버가 없습니다.")

# 최종 정리

```text
AIHub
 └─ 1.Training
      ├─ 원천데이터 TS*.zip
      │       └─ JPG
      │
      └─ 라벨링데이터 TL*.zip
              └─ JSON
                   ↓
          JPG ↔ JSON 매칭
                   ↓
          disease 클래스 확인
                   ↓
       클래스별 최대 300장 추출
                   ↓
       Resize / Normalize / HOG
                   ↓
                 X, y
                   ↓
       Train / Validation / Test
                   ↓
            Linear SVM 학습
                   ↓
           Macro-F1 / 혼동행렬
```

### 핵심

- Validation 원본을 학습에 섞지 않는다.
- AIHub **Training** 데이터에서 수업용 subset을 만든다.
- 대용량 ZIP 전체를 압축 해제하지 않고 필요한 이미지/JSON만 추출한다.
- AIHub의 선택 다운로드는 `filekey`가 가리키는 **ZIP 단위**이므로, 네트워크 다운로드 자체를 600장만으로 제한할 수는 없다.
- 모든 결과는 노트북 실행 폴더 아래 `aihub_plant_project`에 저장된다.
- Windows 경로를 직접 지정할 때는 `Path(r"D:\aihub_plant_project")`처럼 raw 문자열을 사용한다.

### 참고

- [AIHub 식물 병 유발 통합 데이터](https://aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&dataSetSn=525)
- AIHub 공식 페이지는 분할 압축 파일 때문에 Windows에서 WSL 사용을 안내한다. 이 노트북은 같은 과정을 Python으로 처리한다.